# ۳ · تشخیص و جستجوی غذا

پیداکردن یک غذای مشخص در آرشیو — حتی وسط یک سفره‌ی شلوغ.

**معماری:** RT-DETR-X (تشخیص ظرف/غذا) → CLIP ViT-L/14 (تشخیص نوع غذا)

---

## 💡 این ماژول بهترین طراحی پروژه‌ی اصلی را داشت

ایده‌ی **پرامپت‌های رقیب** از همین ماژول آمد و حالا در ماژول حیوان هم
استفاده می‌شود. به‌جای آستانه گذاشتن روی شباهت خام:

```python
text_candidates = [
    user_query,                                     # فرضیه
    "a photo of a completely different food",       # رقیب
    "a photo of an empty plate or background",      # رقیب
]
probs = logits.softmax(dim=-1)
```

**چرا بهتر است؟** شباهت کسینوسی «فرضیه‌ی صفر» ندارد. softmax روی رقبا
به مدل اجازه می‌دهد بگوید «هیچ‌کدام»، پس خروجی یک احتمال کالیبره است.

شاهدش در دموی خودت: این مسیر **۹۸.۸٪** و **۹۶.۹٪** می‌داد، در حالی که
مسیر متنی با کسینوس خام برای بهترین نتیجه‌اش **۲۳.۳٪** نشان می‌داد.
همان مدل، همان فضای برداری — فقط قاعده‌ی تصمیم فرق دارد.

---

## 🐞 باگ‌هایی که رفع شد

| # | مشکل | اثر |
|:--|:--|:--|
| ۱ | فیلتر تداخل با صورت **هرگز اجرا نمی‌شد** | هر کادر غذایی روی صورت انسان پذیرفته می‌شد |
| ۲ | لیست کلاس ایندکس ≠ جستجو | کلاس ۵۶ (**صندلی**) در جستجو برمی‌گشت |
| ۳ | `if device == "cuda"` | روی GPU کرش می‌کرد |

## 🐞 باگ ۱ — دقیق‌تر ببینیم

در `run_advanced_preprocessing` ترتیب اجرا این بود:

```
خط ۱۹۰:  animal_preds = detector_model(...)      ← حیوان
خط ۲۰۳:  food_results = detector(...)            ← غذا
خط ۲۲۰:      if has_face == 1:                   ← ❌ has_face هنوز ۰ است
خط ۲۲۱:          for face in faces:              ← ❌ faces هنوز تعریف نشده
خط ۲۴۸:  faces = face_app.get(img_bgr)           ← چهره (۳۰ خط بعد!)
```

`has_face` در ابتدای هر تکرار صفر می‌شود و چهره‌ها **بعد از** بلوک غذا
محاسبه می‌شوند. پس شرط خط ۲۲۰ **همیشه غلط** بود و کل فیلتری که کامنت
تبلیغش می‌کرد، حتی یک بار اجرا نشد.

**✅ راه‌حل:** ترتیب `چهره → حیوان → غذا`. یک جابه‌جایی بلوک، و قابلیت
زنده می‌شود.

---

## 🐞 باگ ۲ — لیست کلاس‌ها

| | کلاس‌های COCO | conf |
|:--|:--|:--|
| ایندکس | `[39…55]` بطری تا کیک | ۰.۱۵ |
| جستجو | `[45…56]` کاسه تا **صندلی** | ۰.۲۵ |

دو پیامد:

1. کلاس‌های ۳۹–۴۴ (بطری، لیوان، فنجان، چنگال، کارد، قاشق) عکس را
   `has_food=1` می‌کردند ولی در جستجو هیچ برشی تولید نمی‌شد → کاندیدای هدررفته
2. **کلاس ۵۶ صندلی است** — دقیقاً همان که کامنت خودت در ایندکس می‌گفت
   «کلاس 56 (صندلی) حذف شد!». باگی که آگاهانه در ایندکس رفع شده بود،
   در جستجو برگشته بود.

**✅ راه‌حل:** یک منبع واحد + قاعده‌ی `search ⊆ gate`.

In [ ]:
# ── نصب وابستگی‌ها ───────────────────────────────────────────────────
# ترتیب مهم است: insightface نسخه CPU از onnxruntime نصب می‌کند،
# پس اول آن را حذف و بعد نسخه GPU را نصب می‌کنیم.
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q insightface ultralytics transformers opencv-python tqdm
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("✅ نصب کامل شد")
print("موتورهای در دسترس:", ort.get_available_providers())

In [ ]:
# ── تشخیص دستگاه و دقت عددی ─────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی سه جور مقایسه داشت:
#        if device.type == "cuda":   ← درست
#        if device == "cuda":        ← همیشه False !
#
#    چون device یک شیء torch.device است، نه رشته.
#    تست شده روی torch 2.8:  torch.device('cuda') == 'cuda'  →  False
#
#    نتیجه: روی GPU مدل half می‌شد ولی ورودی float32 می‌ماند →
#    RuntimeError: expected scalar type Half but found Float
#
# ✅ راه‌حل: دستگاه و dtype را یکجا حل می‌کنیم تا نتوانند با هم اختلاف پیدا کنند.

import torch
from dataclasses import dataclass


@dataclass(frozen=True)
class Runtime:
    device: torch.device
    use_half: bool

    @property
    def is_cuda(self) -> bool:
        return self.device.type == "cuda"

    def cast_inputs(self, inputs: dict) -> dict:
        """انتقال ورودی به دستگاه با dtype هماهنگ.
        فقط تنسورهای اعشاری half می‌شوند؛ input_ids باید عدد صحیح بماند."""
        out = {}
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                v = v.to(self.device)
                if self.use_half and v.is_floating_point():
                    v = v.half()
            out[k] = v
        return out

    def prepare_model(self, model):
        model = model.to(self.device)
        if self.use_half:
            model = model.half()
        return model.eval()


def resolve_runtime(preference: str = "auto", half: bool = True) -> Runtime:
    name = preference
    if preference == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(name)
    # fp16 روی CPU کندتر از fp32 است و بعضی عملیات پشتیبانی نمی‌شوند
    return Runtime(device=device, use_half=half and device.type == "cuda")


RT = resolve_runtime()
print(f"🚀 دستگاه: {RT.device}   |   نیمه‌دقت (FP16): {RT.use_half}")

In [ ]:
# ── ابزار جعبه‌ها ───────────────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    برش بدون کلمپ مرزی:  image[y1-pad : y2+pad, x1-pad : x2+pad]
#    اگر x1-pad منفی شود، numpy خطا نمی‌دهد — بی‌صدا از ته آرایه
#    برمی‌دارد و برش غلط می‌دهد. یعنی حیوانی که به لبه کادر چسبیده،
#    از روی پیکسل‌های اشتباه طبقه‌بندی می‌شد.

def area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def overlap_fraction(box, other) -> float:
    """چه کسری از box داخل other است.

    عمداً نامتقارن: سؤال «چقدر از این کادرِ غذا روی صورت است»،
    نه «این دو کادر چقدر شبیه‌اند». IoU جواب سؤال اشتباه را می‌دهد —
    کادر کوچک غذا کاملاً داخل کادر بزرگ صورت، IoU پایینی دارد
    ولی overlap_fraction آن ۱.۰ است.
    """
    ax1, ay1, ax2, ay2 = box
    bx1, by1, bx2, by2 = other
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix1 >= ix2 or iy1 >= iy2:
        return 0.0
    a = area(box)
    return ((ix2 - ix1) * (iy2 - iy1) / a) if a else 0.0


def overlaps_any(box, others, threshold: float) -> bool:
    return any(overlap_fraction(box, o) > threshold for o in others)


def pad_box(box, width, height, ratio=0.0, pixels=0):
    """بزرگ‌کردن کادر + کلمپ به مرز تصویر (✅ رفع باگ اسلایس منفی)."""
    x1, y1, x2, y2 = box
    px = int(round((x2 - x1) * ratio)) + pixels
    py = int(round((y2 - y1) * ratio)) + pixels
    return (max(0, x1 - px), max(0, y1 - py),
            min(width, x2 + px), min(height, y2 + py))


def is_large_enough(box, min_px: int) -> bool:
    """رد کردن کادرهای ریز.

    🐞 قبلاً هیچ حداقلی نبود: یک کادر ۱۲×۹ پیکسل ۲۰ برابر بزرگ می‌شد
    به ۲۲۴×۲۲۴ و یک برچسب با درصد بالا می‌گرفت.
    """
    x1, y1, x2, y2 = box
    return (x2 - x1) >= min_px and (y2 - y1) >= min_px


print("✅ ابزار جعبه‌ها آماده (با کلمپ مرزی و حداقل اندازه)")

In [ ]:
# ── امتیازدهی رقابتی ────────────────────────────────────────────────
#
# 💡 بهترین ایده‌ی کل پروژه — و حالا در هر سه مسیر تشخیص استفاده می‌شود.
#
# به‌جای آستانه گذاشتن روی شباهت خام، پرسش را رقابتی می‌کنیم:
#
#     prompts = ["a photo of a zebra",                    ← فرضیه
#                "a photo of a different kind of animal", ← رقیب
#                "a photo of scenery with no animal"]     ← رقیب
#     score = softmax(logits)[0]
#
# چرا بهتر است؟ شباهت کسینوسی «فرضیه صفر» ندارد — هر برشی یک عددی
# می‌دهد و جای برش اصولی ندارد. softmax روی رقبا به مدل اجازه می‌دهد
# بگوید «هیچ‌کدام»، پس خروجی یک احتمال کالیبره است نه یک فاصله.
#
# شاهدش در همین پروژه: مسیر غذا (با رقیب) ۹۶–۹۹٪ می‌داد،
# مسیر متنی (کسینوس خام) برای بهترین نتیجه‌اش ۲۳٪ نشان می‌داد.

import math
from dataclasses import dataclass


def softmax(values):
    if not values:
        return []
    top = max(values)
    exps = [math.exp(v - top) for v in values]   # پایدار عددی
    total = sum(exps)
    return [e / total for e in exps]


def contrastive_score(logits) -> float:
    """احتمال پسین فرضیه در برابر رقبایش."""
    if not logits:
        raise ValueError("logits خالی")
    if len(logits) == 1:
        raise ValueError("حداقل یک prompt رقیب لازم است؛ "
                         "با یک کاندیدا softmax همیشه ۱.۰ است")
    return softmax(logits)[0]


@dataclass(frozen=True)
class Match:
    file_name: str
    file_path: str
    score: float
    box: tuple = None
    label: str = None


def best_per_image(matches):
    """یک نتیجه به‌ازای هر تصویر — قوی‌ترین نمونه.

    🐞 باگی که رفع شد:
       نسخه یکپارچه break هر تصویر را از دست داده بود، پس عکسی با
       دو چهره‌ی منطبق دو بار در نتایج ظاهر می‌شد.

       ضمناً کد قبلی «اولین» نمونه را ثبت می‌کرد و همان را کلید
       مرتب‌سازی می‌کرد — پس عکسی که گربه دومش ۹۵٪ بود با ۲۶٪
       رتبه‌بندی می‌شد.
    """
    best = {}
    for m in matches:
        cur = best.get(m.file_name)
        if cur is None or m.score > cur.score:
            best[m.file_name] = m
    return sorted(best.values(), key=lambda m: m.score, reverse=True)


assert abs(sum(softmax([1.0, 2.0, 3.0])) - 1.0) < 1e-9
assert contrastive_score([10.0, 0.0, 0.0]) > 0.95
assert contrastive_score([0.0, 10.0, 0.0]) < 0.05
print("✅ امتیازدهی رقابتی آماده")

## ⚙️ تنظیمات

In [ ]:
from dataclasses import dataclass

# نام کلاس‌های COCO برای خوانایی — تا دوباره کسی ۵۶ را اشتباه نگیرد
COCO = {39:"bottle", 40:"wine glass", 41:"cup", 42:"fork", 43:"knife",
        44:"spoon", 45:"bowl", 46:"banana", 47:"apple", 48:"sandwich",
        49:"orange", 50:"broccoli", 51:"carrot", 52:"hot dog", 53:"pizza",
        54:"donut", 55:"cake", 56:"chair"}     # ← ۵۶ صندلی است، غذا نیست

@dataclass(frozen=True)
class FoodConfig:
    gallery: str = "/content/gallery"
    database: str = "food_index.db"

    # گیت (زمان ایندکس): سخاوتمند — ظرف و قاشق هم نشانه‌ی صحنه‌ی غذا هستند
    gate_classes: tuple = (39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55)
    # جستجو: فقط چیزهایی که می‌توانند «خودِ غذا» باشند
    search_classes: tuple = (45,46,47,48,49,50,51,52,53,54,55)

    gate_confidence: float = 0.15
    search_confidence: float = 0.25

    min_crop_px: int = 50
    padding_px: int = 20        # پیکسل ثابت، نه درصدی — هویت غذا در محتویاتش است

    match_threshold: float = 0.60
    negative_prompts: tuple = (
        "a photo of a completely different food",
        "a photo of an empty plate or background",
    )

    # اگر بیش از این کسر از کادر غذا روی صورت باشد، ردش کن
    face_overlap_reject: float = 0.50

    def prompts_for(self, query):
        return [query, *self.negative_prompts]

CFG = FoodConfig()

# ✅ قواعدی که باگ ۲ را غیرممکن می‌کنند
assert set(CFG.search_classes) <= set(CFG.gate_classes), \
    "کلاسی جستجو می‌شود که هرگز ایندکس نشده"
assert 56 not in CFG.gate_classes and 56 not in CFG.search_classes, \
    "کلاس ۵۶ صندلی است، غذا نیست"
assert CFG.gate_confidence <= CFG.search_confidence, \
    "گیت نباید سخت‌گیرتر از جستجو باشد"

print("✅ تنظیمات معتبر")
print("   گیت   :", [COCO[c] for c in CFG.gate_classes])
print("   جستجو :", [COCO[c] for c in CFG.search_classes])

## 🧠 بارگذاری مدل‌ها

In [ ]:
import torch
from insightface.app import FaceAnalysis
from transformers import CLIPModel, CLIPProcessor
from ultralytics import RTDETR

detector = RTDETR("rtdetr-x.pt")
print("✅ RT-DETR-X")

CLIP_NAME = "openai/clip-vit-large-patch14"
clip_model = RT.prepare_model(CLIPModel.from_pretrained(CLIP_NAME))
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
print("✅ CLIP ViT-L/14 —", "FP16" if RT.use_half else "FP32")

# لازم برای فیلتر تداخل با صورت (باگ ۱)
providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
             if RT.is_cuda else ["CPUExecutionProvider"])
face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if RT.is_cuda else -1, det_size=(640, 640))
print("✅ InsightFace (برای فیلتر تداخل)")

## 📥 ایندکس‌گذاری — با ترتیب درست

**✅ رفع باگ ۱:** چهره **اول** تشخیص داده می‌شود، بعد غذا. حالا فیلتر
تداخل واقعاً چیزی برای بررسی دارد.

In [ ]:
import sqlite3
from contextlib import contextmanager
from pathlib import Path

import cv2
from tqdm.auto import tqdm

SCHEMA = """
CREATE TABLE IF NOT EXISTS gallery_meta (
    file_name TEXT PRIMARY KEY,
    file_path TEXT NOT NULL,
    has_face  INTEGER NOT NULL DEFAULT 0,
    has_food  INTEGER NOT NULL DEFAULT 0
);
CREATE INDEX IF NOT EXISTS idx_has_food ON gallery_meta(has_food);
"""

@contextmanager
def connect():
    conn = sqlite3.connect(CFG.database)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()

with connect() as c:
    c.executescript(SCHEMA)


def detect_food(image_or_path, classes, confidence):
    res = detector(image_or_path, classes=list(classes), conf=confidence, verbose=False)
    out = []
    for r in res:
        for b in r.boxes:
            x1, y1, x2, y2 = (int(v) for v in b.xyxy[0])
            out.append(((x1, y1, x2, y2), float(b.conf[0])))
    return out


def analyse_image(img_bgr, path):
    """ترتیب اجرا: چهره ← حیوان ← غذا. این ترتیب load-bearing است."""
    h, w = img_bgr.shape[:2]

    # ═══ ۱. چهره — باید اول باشد، چون مرحله غذا به آن نیاز دارد ═══
    faces = face_app.get(img_bgr)
    face_boxes = [tuple(int(v) for v in f.bbox[:4]) for f in faces]

    # ═══ ۲. غذا — حالا face_boxes موجود است ═══
    kept, rejected = [], []
    for box, conf in detect_food(str(path), CFG.gate_classes, CFG.gate_confidence):
        if not is_large_enough(box, CFG.min_crop_px):
            continue
        # ✅ فیلتری که قبلاً هرگز اجرا نمی‌شد
        if overlaps_any(box, face_boxes, CFG.face_overlap_reject):
            rejected.append(box)
            continue
        kept.append(box)

    return {"faces": face_boxes, "food": kept, "rejected_food": rejected}


def build_food_index():
    paths = sorted(p for p in Path(CFG.gallery).iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    total_rejected = 0
    with connect() as conn:
        for path in tqdm(paths, desc="Indexing"):
            img = cv2.imread(str(path))
            if img is None:
                continue
            a = analyse_image(img, path)
            total_rejected += len(a["rejected_food"])
            conn.execute("INSERT OR REPLACE INTO gallery_meta VALUES (?,?,?,?)",
                         (path.name, str(path),
                          int(len(a["faces"]) > 0), int(len(a["food"]) > 0)))
    print("✅ ایندکس تمام شد")
    print("   ", total_rejected, "کادر غذا به‌خاطر تداخل با صورت رد شد")
    print("    (در نسخه قبلی این عدد همیشه صفر بود — فیلتر اجرا نمی‌شد)")

# build_food_index()

## 🔎 جستجو

In [ ]:
import time

from PIL import Image

def score_crops(crops, prompts):
    inputs = clip_processor(text=list(prompts), images=list(crops),
                            return_tensors="pt", padding=True)
    # ✅ رفع باگ ۳: dtype از RT می‌آید، نه از مقایسه‌ی شکسته‌ی رشته‌ای
    inputs = RT.cast_inputs(dict(inputs))
    with torch.no_grad():
        return clip_model(**inputs).logits_per_image.float().cpu().tolist()


def search_food(query, threshold=None):
    threshold = CFG.match_threshold if threshold is None else threshold
    t0 = time.time()

    with connect() as conn:
        rows = conn.execute(
            "SELECT file_name, file_path FROM gallery_meta WHERE has_food = 1"
        ).fetchall()
    print("پیش‌فیلتر:", len(rows), "تصویر کاندیدا")

    candidates, crops = [], []
    for name, path in rows:
        img = cv2.imread(path)
        if img is None:
            continue
        h, w = img.shape[:2]
        # ✅ همان منبع کلاس‌ها که ایندکس استفاده کرد — صندلی وارد نمی‌شود
        for box, conf in detect_food(path, CFG.search_classes, CFG.search_confidence):
            if not is_large_enough(box, CFG.min_crop_px):
                continue
            pb = pad_box(box, w, h, pixels=CFG.padding_px)   # پدینگ + کلمپ
            patch = img[pb[1]:pb[3], pb[0]:pb[2]]
            if patch.size == 0:
                continue
            candidates.append((name, path, box))
            crops.append(Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)))

    if not crops:
        print("هیچ برشی پیدا نشد")
        return []

    logits = score_crops(crops, CFG.prompts_for(query))

    matches = []
    for (name, path, box), row in zip(candidates, logits):
        score = contrastive_score(row)
        if score >= threshold:
            matches.append(Match(name, path, score, box=box, label=query))

    results = best_per_image(matches)
    print("جستجو در", round(time.time() - t0, 3), "ثانیه —", len(results), "نتیجه")
    for i, m in enumerate(results, 1):
        print(" [%2d] %-28s %6.2f%%" % (i, m.file_name, m.score * 100))
    return results

# search_food("French fries")

## 📋 خلاصه‌ی این ماژول

| | قبلاً | الان |
|:--|:--|:--|
| ترتیب مراحل | حیوان → غذا → چهره | **چهره → حیوان → غذا** |
| فیلتر تداخل با صورت | نوشته شده ولی **هرگز اجرا نشد** | واقعاً اجرا می‌شود |
| کلاس‌های ایندکس | `[39…55]` | یک منبع واحد |
| کلاس‌های جستجو | `[45…56]` ← **صندلی** | زیرمجموعه‌ی گیت، بدون ۵۶ |
| `if device == "cuda"` | همیشه False → کرش روی GPU | `RT.cast_inputs` |
| حداقل اندازه برش | فقط در جستجو | هر دو مرحله |

**آنچه نگه داشتیم:** ایده‌ی پرامپت‌های رقیب — که حالا الگوی مشترک
سه مسیر از چهار مسیر جستجوست.